# 01 - Exploratory Data Analysis

Fake job postings (EMSCAD / Kaggle `shivamb/real-or-fake-fake-jobposting-prediction`).
~17.9k postings, of which roughly 5% are fraudulent.

**Read this first.** A model that predicts "real" for every single posting scores
about **95% accuracy** and catches **zero** scams. It is useless and it looks
excellent. That is why this project is scored on **average precision (PR-AUC)**,
which only rewards ranking the rare fraudulent class correctly.

EDA runs on the **training split only**. Val and test stay sealed until the models
are scored, so nothing we learn here can leak into them.

Prerequisite: `python -m src.data` then `python -m src.preprocessing`.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether this runs from notebooks/ or the project root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, evaluation, features, models, preprocessing

config.set_seed()
config.ensure_dirs()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
train, _, _ = preprocessing.load_splits()
print(f"{len(train):,} training postings, {train.shape[1]} columns")
print(f"fraudulent: {train[config.TARGET].sum():,} ({train[config.TARGET].mean():.2%})")
train.head(3)

## 1. The target is heavily imbalanced

The first plot is the whole reason for every methodological choice that follows.

In [ ]:
counts = train[config.TARGET].value_counts().sort_index()
majority_share = counts.max() / counts.sum()

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(["real (0)", "fake (1)"], counts.values, color=["#4c72b0", "#c44e52"])
for bar, value in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:,}\n({value / counts.sum():.1%})",
            ha="center", va="bottom")
ax.set_ylabel("postings")
ax.set_title("Target distribution (train split)")
ax.set_ylim(0, counts.max() * 1.15)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "01_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f'"always real" baseline accuracy: {majority_share:.2%}')
print(f'"always real" scams caught:      0 of {counts.get(1, 0):,}')
print(f"a random ranker scores AP = {train[config.TARGET].mean():.4f} - that is the number to beat")

> **Insight 1.** Accuracy is unusable here: guessing "real" every time wins ~95%
> while catching nothing. Average precision for a random ranker equals the positive
> rate (~0.05), so that is the real floor.

## 2. Missingness is the signal

Scammers do not fill in the boring fields. Compare how often each field is empty
for fake vs real postings - this is the single most useful thing in the dataset.

In [ ]:
flags = ["company_profile_is_missing", "salary_range_is_missing", "benefits_is_missing",
         "requirements_is_missing", "department_is_missing"]
rates = train.groupby(config.TARGET)[flags].mean().T
# has_company_logo / has_questions are presence flags, so invert them to read as "missing".
rates.loc["no_company_logo"] = 1 - train.groupby(config.TARGET)["has_company_logo"].mean()
rates.loc["no_screening_questions"] = 1 - train.groupby(config.TARGET)["has_questions"].mean()
rates.columns = ["real", "fake"]
rates = rates.sort_values("fake", ascending=True)

ax = rates.plot.barh(figsize=(8, 5), color=["#4c72b0", "#c44e52"])
ax.set_xlabel("share of postings where the field is absent")
ax.set_title("Missingness by class (train split)")
ax.set_xlim(0, 1)
ax.legend(title="class")
ax.figure.tight_layout()
ax.figure.savefig(config.FIGURES_DIR / "02_missingness_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

lift = (rates["fake"] / rates["real"].replace(0, np.nan)).sort_values(ascending=False)
print("How much more often a field is missing for fakes (ratio):")
print(lift.round(2).to_string())

> **Insight 2.** Fraudulent postings are far more likely to have no company profile
> and no company logo. This is exactly why `src/features.py` builds an explicit
> `*_is_missing` flag for every one of these fields instead of quietly imputing
> them - the absence *is* the evidence.

## 3. Text length separates the classes

Real postings are written by someone who wants the role filled. Scams are thinner.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, name, colour in [(0, "real", "#4c72b0"), (1, "fake", "#c44e52")]:
    subset = train.loc[train[config.TARGET] == label, "full_text_word_count"]
    axes[0].hist(subset, bins=60, range=(0, 1200), alpha=0.55, density=True,
                 label=name, color=colour)
axes[0].set_xlabel("words in full_text")
axes[0].set_ylabel("density")
axes[0].set_title("Total posting length")
axes[0].legend()

length_cols = ["company_profile_word_count", "description_word_count",
               "requirements_word_count", "benefits_word_count"]
melted = train.melt(id_vars=config.TARGET, value_vars=length_cols,
                    var_name="field", value_name="words")
melted["field"] = melted["field"].str.replace("_word_count", "", regex=False)
melted["class"] = melted[config.TARGET].map({0: "real", 1: "fake"})
sns.boxplot(data=melted, x="field", y="words", hue="class", ax=axes[1], showfliers=False,
            palette={"real": "#4c72b0", "fake": "#c44e52"})
axes[1].set_title("Length per field")
axes[1].set_ylabel("words")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "03_text_length_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

print(train.groupby(config.TARGET)[["full_text_word_count", "caps_ratio",
                                    "exclamation_count"]].median().round(3).to_string())

> **Insight 3.** Fakes skew shorter overall, and the gap is widest on
> `company_profile`. The medians for `caps_ratio` and `exclamation_count` are worth
> a glance too - shouting is cheap.

## 4. What the two classes actually say

Top bigrams by class, then word clouds. `wordcloud` is optional - the cell degrades
to the bar chart if it is not installed.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


def top_ngrams(texts, n=15, ngram_range=(2, 2)):
    """Most frequent n-grams in a set of documents."""
    vec = CountVectorizer(ngram_range=ngram_range, stop_words="english", min_df=3)
    matrix = vec.fit_transform(texts)
    freqs = np.asarray(matrix.sum(axis=0)).ravel()
    order = np.argsort(-freqs)[:n]
    names = vec.get_feature_names_out()
    return pd.Series(freqs[order], index=names[order])


fake_text = train.loc[train[config.TARGET] == 1, config.FULL_TEXT_COLUMN]
real_text = train.loc[train[config.TARGET] == 0, config.FULL_TEXT_COLUMN]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_ngrams(fake_text).sort_values().plot.barh(ax=axes[0], color="#c44e52")
axes[0].set_title("Top bigrams - FAKE")
top_ngrams(real_text).sort_values().plot.barh(ax=axes[1], color="#4c72b0")
axes[1].set_title("Top bigrams - REAL")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "04_top_ngrams.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
try:
    from wordcloud import WordCloud
except ImportError:
    print("wordcloud not installed - skipping (pip install wordcloud). Bigram chart above covers it.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, texts, title in [(axes[0], fake_text, "FAKE"), (axes[1], real_text, "REAL")]:
        cloud = WordCloud(width=800, height=400, background_color="white",
                          stopwords=None, random_state=config.SEED,
                          collocations=False).generate(" ".join(texts.head(2000)))
        ax.imshow(cloud, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Word cloud - {title}")
    fig.tight_layout()
    fig.savefig(config.FIGURES_DIR / "05_wordclouds.png", dpi=150, bbox_inches="tight")
    plt.show()

> **Insight 4.** Fake postings cluster around data entry, admin and work-from-home
> language, plus earnings promises. Real ones read like ordinary corporate job ads.
> A bag-of-words model should already do well - which is exactly what the baseline
> in notebook 02 tests.

## 5. Fraud rate by category

Absolute counts hide the story here; the fraud *rate* within each category is what
matters, so both are plotted together.

In [ ]:
def fraud_rate_by(column, min_count=30, top=12):
    """Fraud rate per category, keeping only categories with enough support."""
    grouped = train.groupby(column)[config.TARGET].agg(["mean", "count"])
    grouped = grouped[grouped["count"] >= min_count]
    return grouped.sort_values("mean", ascending=False).head(top)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
overall = train[config.TARGET].mean()

for ax, column in zip(axes, ["employment_type", "required_experience", "industry"]):
    table = fraud_rate_by(column).sort_values("mean")
    ax.barh(table.index.astype(str), table["mean"], color="#c44e52")
    ax.axvline(overall, color="black", linestyle="--", linewidth=1,
               label=f"overall {overall:.1%}")
    for y, (rate, count) in enumerate(zip(table["mean"], table["count"])):
        ax.text(rate, y, f"  n={count:,}", va="center", fontsize=7)
    ax.set_title(f"Fraud rate by {column}")
    ax.set_xlabel("fraud rate")
    ax.legend(fontsize=7)

fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "06_fraud_rate_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

print("Highest-risk industries (min 30 postings):")
print(fraud_rate_by("industry").assign(mean=lambda d: d["mean"].round(3)).to_string())

## Takeaways

1. **~5% positives.** Accuracy is meaningless; the headline metric is average
   precision, with ROC-AUC and minority-class F1/precision/recall alongside.
2. **Missingness is the strongest cheap signal** - no company profile, no logo.
   Encoded explicitly as `*_is_missing` features rather than imputed away.
3. **Length and tone differ**: fakes are shorter, shoutier, thinner on the company
   profile.
4. **Vocabulary differs sharply**, so TF-IDF should be a strong baseline and the
   transformer models have something real to learn.
5. **Risk concentrates in a few industries and in part-time / entry-level roles**,
   so the categorical fields earn their place next to the text.

A default 0.5 threshold is wrong for a problem this imbalanced. Every notebook from
here on tunes the threshold on validation and only then reports on test.

Next: `02_baseline.ipynb`.